#1. Dataset

In [1]:
import torch

with open("t8.shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Character-level vocab
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

#2.Positional Encoding

In [2]:
import math
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model))

        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

#3.Multihead attention

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.qkv = nn.Linear(d_model, d_model * 3)
        self.fc = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, C = x.shape

        qkv = self.qkv(x)
        qkv = qkv.reshape(B, T, 3, self.n_heads, self.d_k)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn = torch.softmax(scores, dim=-1)
        out = attn @ v

        out = out.transpose(1, 2).contiguous().reshape(B, T, C)
        return self.fc(out)

#4.Feed Forward

In [4]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)

#5.Encoder Layer

In [5]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x

#6. Decoder Layer (with masking)

In [6]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, enc_out, mask):
        x = x + self.self_attn(self.norm1(x), mask)
        x = x + self.cross_attn(self.norm2(x))
        x = x + self.ff(self.norm3(x))
        return x

#7. Full Transformer

In [7]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=4):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)

        self.encoder = nn.ModuleList(
            [EncoderLayer(d_model, n_heads) for _ in range(n_layers)]
        )

        self.decoder = nn.ModuleList(
            [DecoderLayer(d_model, n_heads) for _ in range(n_layers)]
        )

        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt, mask):
        src = self.pos(self.embed(src))
        tgt = self.pos(self.embed(tgt))

        for layer in self.encoder:
            src = layer(src)

        for layer in self.decoder:
            tgt = layer(tgt, src, mask)

        return self.fc(tgt)

#8.Masking

In [8]:
def generate_mask(size):
    mask = torch.tril(torch.ones(size, size))
    return mask.unsqueeze(0).unsqueeze(0)

#9. Training Loop

In [ ]:
from tqdm import tqdm

model = Transformer(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

seq_len = 64

for epoch in range(10):
    print(f"\nEpoch {epoch+1}")

    # wrap loop with tqdm
    loop = tqdm(range(0, len(data)-seq_len-1, seq_len), desc="Training")

    for i in loop:
        src = data[i:i+seq_len].unsqueeze(0)
        tgt = data[i+1:i+seq_len+1].unsqueeze(0)

        mask = generate_mask(seq_len)

        out = model(src, tgt, mask)
        loss = loss_fn(out.view(-1, vocab_size), tgt.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # update progress bar with loss
        loop.set_postfix(loss=loss.item())


Epoch 1


Training: 100%|██████████| 85284/85284 [2:15:47<00:00, 10.47it/s, loss=0]



Epoch 2


Training:  90%|████████▉ | 76711/85284 [2:04:39<13:40, 10.44it/s, loss=0]